<a href="https://colab.research.google.com/github/realnanayawjnr/demo-repo/blob/main/Work_with_ChatGPT_Assistance_for_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
# mount my google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
# change my directory to works folder
%cd /content/drive/MyDrive/'Youth in AI'/'ULTRASOUND LIVER IMAGE SEGMENTATION'/

/content/drive/MyDrive/Youth in AI/ULTRASOUND LIVER IMAGE SEGMENTATION


Great! Here's a complete Python script you can run on your local machine to:

    Extract your dataset from a ZIP file.

    Preprocess the images (resize and normalize).

    Split into train, validation, and test sets.

    Train two models: CNN (Keras) and Random Forest.

    Evaluate and select the best model.

    Test and present a conclusion.

✅ Requirements

Install these libraries if you haven't yet:

In [36]:
# Set up Notebook
%matplotlib inline

# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
import cv2
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical


# We do this to ignore several specific warnings
import warnings
warnings.filterwarnings("ignore")

sns.set(style="white")


In [37]:
# Define the base directory for the image data.  This should point to the folder containing your dataset.
base_dir = "/content/drive/MyDrive/Youth in AI/ULTRASOUND LIVER IMAGE SEGMENTATION/Data/7272660" # Corrected path

# Define the categories of ultrasound images (Benign, Malignant, Normal).
categories = ["Benign", "Malignant", "Normal"]

# Initialize an empty list to store the data.
data = []

# Iterate through each category of images.
for category in categories:

    # Construct the paths to the image and segmentation files for the current category.
    image_dir = os.path.join(base_dir, category, category, "image")
    seg_dir = os.path.join(base_dir, category, category, "segmentation", "liver")

    # Get a list of image files (PNG, JPG, JPEG) and sort them for consistency.
    image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    # Get a list of segmentation JSON files and sort them for consistency.
    json_files = sorted([f for f in os.listdir(seg_dir) if f.lower().endswith('.json')])

    # Iterate through image and segmentation files, ensuring they correspond correctly.
    for img, jsn in zip(image_files, json_files):

        # Create a dictionary for each image and its associated segmentation data.
        data.append({
            "Category": category, # Category of the image (Benign, Malignant, Normal)
            "Image_Path": os.path.join(image_dir, img), # Full path to the image file
            "Segmentation_JSON_Path": os.path.join(seg_dir, jsn) # Full path to the segmentation JSON file
        })

# Create a Pandas DataFrame from the collected data.
df = pd.DataFrame(data)


In [38]:
# --- Step 2: Load and preprocess the images ---
def load_images(df, img_size=(64, 64)):
    X, y = [], []
    labels = df['Category'].unique()
    label_map = {label: idx for idx, label in enumerate(labels)}
    for index, row in df.iterrows():
        img_path = row['Image_Path']
        try:
            img = cv2.imread(img_path)
            img = cv2.resize(img, img_size)
            img = img / 255.0  # normalize
            X.append(img)
            y.append(label_map[row['Category']])
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            pass  # skip unreadable images
    return np.array(X), np.array(y), label_map

X, y, label_map = load_images(df)
print(f"Loaded {len(X)} images across {len(label_map)} classes")


Loaded 732 images across 3 classes


In [39]:
# --- Step 3: Split the data ---
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [40]:
# --- Step 4a: CNN Model ---
def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn(X_train.shape[1:], len(label_map))
cnn_model.fit(X_train, to_categorical(y_train), epochs=5, validation_data=(X_val, to_categorical(y_val)))


Epoch 1/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 200ms/step - accuracy: 0.5362 - loss: 1.0133 - val_accuracy: 0.6545 - val_loss: 0.8400
Epoch 2/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 205ms/step - accuracy: 0.7176 - loss: 0.7098 - val_accuracy: 0.6364 - val_loss: 0.7403
Epoch 3/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 169ms/step - accuracy: 0.7524 - loss: 0.5923 - val_accuracy: 0.6818 - val_loss: 0.6713
Epoch 4/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 170ms/step - accuracy: 0.7791 - loss: 0.5144 - val_accuracy: 0.6909 - val_loss: 0.6310
Epoch 5/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 288ms/step - accuracy: 0.8384 - loss: 0.4275 - val_accuracy: 0.7091 - val_loss: 0.6015


In [41]:
# --- Step 4c: Random Forest with Flattened Images ---
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)

rf = RandomForestClassifier()
rf.fit(X_train_flat, y_train)
rf_val_acc = accuracy_score(y_val, rf.predict(X_val_flat))


In [42]:
# --- Step 5: Compare Models ---
cnn_val_acc = cnn_model.evaluate(X_val, to_categorical(y_val), verbose=0)[1]
print("\nValidation Accuracies:")
print(f"CNN: {cnn_val_acc:.2f}")
print(f"Random Forest: {rf_val_acc:.2f}")



Validation Accuracies:
CNN: 0.71
Random Forest: 0.65


In [43]:
# --- Step 6: Test Best Model ---
best_model = "CNN" if cnn_val_acc > rf_val_acc else "RF"

print(f"\nBest performing model: {best_model}")

if best_model == "CNN":
    test_acc = cnn_model.evaluate(X_test, to_categorical(y_test), verbose=0)[1]
elif best_model == "SVM":
    X_test_hog = extract_hog_features(X_test)
    test_acc = accuracy_score(y_test, svm.predict(X_test_hog))
else:
    X_test_flat = X_test.reshape(len(X_test), -1)
    test_acc = accuracy_score(y_test, rf.predict(X_test_flat))

print(f"Test Accuracy of best model ({best_model}): {test_acc:.2f}")



Best performing model: CNN
Test Accuracy of best model (CNN): 0.70


📌 What’s Happening

    Images are normalized and resized to 64x64.

    CNN is trained using TensorFlow/Keras.

    Random Forest uses flattened pixel data.